# Data Preprocessing Project - G9
## City Hospital

### Group members-
- Rasel Muhammad
- Umar Mansoor
- Aakarshak Asiwal
- Fatima Zahra

In [31]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
import plotly as pty
import datetime as dt

Loading the dataset

In [ ]:
df=pd.read_excel("../data/Project Data.xlsx")
df.head()

,Patient ID,Age,Gender,City of Residence,Profession,Insurance Provider,Family History,Education Level,Marital Status,Visit Date,Department,Consultation Duration,Satisfaction Level,Approximate Annual Income,Consultation Price,Insurance Coverage
0,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-02-22,Endocrinology,110,2,38839.0,293.014042,0.0
1,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-01-21,Emergency,76,4,52219.0,287.208982,0.0
2,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-06-13,Neurology,17,5,41491.0,241.254507,0.0
3,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-03-24,Emergency,15,3,34944.0,221.177483,0.0
4,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-03-19,Neurology,99,1,39248.0,219.743794,0.0


In [26]:
df.shape

(10008, 16)

Since a lot of patients are repeated, let's create a multi-indexed table, grouping first with Patient ID, then individual records. Also, let's do this in ascending order

In [34]:
dcopy=df.copy()

In [35]:
dcopy.set_index(['Patient ID', 'Visit Date']).sort_index(ascending=True)
dcopy.head(10)

,Patient ID,Age,Gender,City of Residence,Profession,Insurance Provider,Family History,Education Level,Marital Status,Visit Date,Department,Consultation Duration,Satisfaction Level,Approximate Annual Income,Consultation Price,Insurance Coverage
0,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-02-22,Endocrinology,110,2,38839.0,293.014042,0.0
1,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-01-21,Emergency,76,4,52219.0,287.208982,0.0
2,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-06-13,Neurology,17,5,41491.0,241.254507,0.0
3,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-03-24,Emergency,15,3,34944.0,221.177483,0.0
4,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-03-19,Neurology,99,1,39248.0,219.743794,0.0
5,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-06-19,Orthopedics,30,3,49086.0,217.654967,0.0
6,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-02-29,Emergency,82,4,47905.0,203.152041,0.0
7,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-01-16,Orthopedics,80,3,52349.0,202.982209,0.0
8,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-02-06,Psychiatry,61,3,55539.0,194.411650,0.0
9,1482,65.0,Female,Birmingham,Retired,NaN,Heart Disease,Master,Divorced,2024-02-03,Rheumatology,86,5,44110.0,192.509241,0.0


Now, let's do some initial cleaning

In [38]:
dcopy.dtypes

Patient ID                            int64
Age                                 float64
Gender                                  str
City of Residence                       str
Profession                              str
Insurance Provider                      str
Family History                          str
Education Level                         str
Marital Status                          str
Visit Date                   datetime64[us]
Department                              str
Consultation Duration                 int64
Satisfaction Level                    int64
Approximate Annual Income           float64
Consultation Price                  float64
Insurance Coverage                  float64
dtype: object

consultation duration's unit is most likely minutes, dont need to convert to datetime. Might do later if scaling issues arise.

In [36]:
dcopy.isnull().sum()

Patient ID                      0
Age                            56
Gender                          0
City of Residence               0
Profession                      0
Insurance Provider           1910
Family History               1993
Education Level                29
Marital Status                  0
Visit Date                      0
Department                      0
Consultation Duration           0
Satisfaction Level              0
Approximate Annual Income     154
Consultation Price              0
Insurance Coverage             50
dtype: int64

Initial thoughts (aak)

age is a critical factor, we would need to dive into those records. Or they could be missing at accident, which we could easily interpolate by looking at patient age from past consulations.

for insurance coverage, let's check if the missing values are also missing from the insurance provider (which could explain that they dont have insurance). Just to be sure, we could map them against the average income.

family history, could either be explained that they don't have any or dont know any. There could be a rare case that they're distancing themselves from their family, but that's a separate case. We could run a surname-by-surname check, but that doesnt guarantee that people are actually related to each other as family, so interpolation is not good for that. We could rather interpolate via exploring if patients consulting a department have a high commonality in family histories or not.

Education level is most likely explained by the patient not wanting to share it, and it's small enough a population to not interfere with nominal exploration. Still, we could run a Box-Cox to cement that fact.

Approximate annual income is the same as education level, but we can interpolate using insurance coverage, profession, and/or age. We could also use education level for this.

Finally insurance provider, I have no clue how to go about this one.

In [39]:
dcopy.describe()

,Patient ID,Age,Visit Date,Consultation Duration,Satisfaction Level,Approximate Annual Income,Consultation Price,Insurance Coverage
count,10008.000000,9952.000000,10008,10008.000000,10008.000000,9.854000e+03,10008.000000,9958.000000
mean,1251.580735,50.635651,2024-03-31 14:53:14.244604,67.807654,3.202338,4.340276e+04,187.263022,115.429366
min,1000.000000,0.000000,2024-01-01 00:00:00,15.000000,1.000000,0.000000e+00,50.036762,0.000000
25%,1127.000000,23.000000,2024-02-15 00:00:00,41.000000,2.000000,3.115775e+04,111.316562,59.692518
50%,1252.000000,52.000000,2024-03-31 00:00:00,68.000000,3.000000,4.087650e+04,159.531115,115.938998
75%,1375.000000,75.000000,2024-05-16 00:00:00,94.000000,5.000000,5.072175e+04,199.820999,165.437958
max,1499.000000,195.000000,2024-06-30 00:00:00,600.000000,6.000000,1.197090e+07,39999.222018,421.887836
std,144.576421,31.185608,NaN,32.447143,1.565507,2.681420e+05,862.865550,79.337759


Some errors found-

Satisfaction level, according to the project guidelines, should be hard-capped to 5 at max. We see the max in that column is 6. Need to reset those values to 5.

Consultation price- the max price listed is goddamn 40 thousand euros. This aint 'murica lil bro.

Annual Income- we got gill bates.

Consultation duration- the max is TEN HOURS? Damn bruh that's genuine patient neglect.

Age- we got count dragula over here.

In [45]:
df.describe(include=['str'])

,Gender,City of Residence,Profession,Insurance Provider,Family History,Education Level,Marital Status,Department
count,10008,10008,10008,8098,8015,9979,10008,10008
unique,3,8,10,4,4,8,4,13
top,Other,Birmingham,Retired,Provider D,Heart Disease,Undergraduate,Divorced,Psychiatry
freq,3417,1473,3601,2165,2235,4160,2893,1361


Summary-

Gender- the top gender that comes is "other", meaning people may be choosing to not report theirs or, in a rare off chance, it's just full of blue-haired baddies.

Tbf they're only about 34% of the data, so it's still an even split.

Most people come from BARMINGAM INNIT, and mostly retirees (14% birmingham, 36% retired)
Note that here, Retired is the dominating class, cause age causes problems.

22.5% have a history of heart disease, and 21.8% of the patients' insurance provider is provider D, which is an even split for it.
Even for familt history, it's about an even split.

There's four categories of martital status split almost evenly, though divorced has a high frequency.

### NOTE-
A thing to notice- most people are undergrads, and massively dominate the records, accounting for 42% of them, out of 8 education levels. I mean, it's cause undergrads are in massive supply in developed (and some developing) nations.

## NOW

some department-level aggregation statistics.

In [43]:
depSummary=dcopy.groupby('Department').agg(
    avgDuration=('Consultation Duration', 'mean'),
    avgSatisfaction=('Satisfaction Level', 'mean'),
    avgPrice=('Consultation Price', 'mean'),
    totalVisits=('Patient ID', 'count')
    )

depSummary

,avgDuration,avgSatisfaction,avgPrice,totalVisits
Department,,,,
Allergology,67.086181,3.062407,148.566610,673
Cardiology,68.557632,2.979751,268.070511,642
Dermatology,67.043873,3.046899,148.649382,661
ENT,67.494640,4.130168,74.709961,653
Emergency,68.364568,2.964861,473.569990,683
Endocrinology,67.284058,3.046377,262.398238,690
Gastroenterology,67.606322,3.091954,149.709256,696
General Practice,66.384673,3.965440,75.291117,1331
Neurology,69.823188,2.943478,302.959055,690


We see that all departments have a near-equal spread of visits, BUT
General Practice and Psychiatry have DOUBLE the visits of the other on average.
What's more interesting is, Psychiatry outnumbers even general practice.

## NOW-

Let's do some department-level aggregations and statistics. Afterwards, let's finally deal with missing data, plot outliers and deal with those

Then, we'll run a correlation heatmap, plot some graphs to check correlations.